In [1]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
!pip install transformers==4.57.1
!pip install --no-deps trl==0.22.2
!pip install wandb weave

In [2]:
import os
os.environ['WANDB_API_KEY'] = "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"

In [3]:
import torch
from datasets import load_dataset
from unsloth import FastVisionModel
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig

# Config
MODEL_NAME = "unsloth/Qwen3-VL-4B-Instruct"
DATASET_NAME = "barryallen16/bike_parts_counterfeit_detection"
OUTPUT_DIR = "./counterfeit_detector"


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


2026-02-05 16:22:02.266463: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770308522.643379      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770308522.751821      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1770308523.228260      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770308523.228301      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770308523.228303      55 computation_placer.cc:177] computation placer alr

🦥 Unsloth Zoo will now patch everything to make training faster!


In [4]:

model, tokenizer = FastVisionModel.from_pretrained(
    MODEL_NAME,
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)

model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=True,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
    target_modules="all-linear",
)

print("✅ Model loaded and configured!")

==((====))==  Unsloth 2026.1.4: Fast Qwen3_Vl patching. Transformers: 4.57.1.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/4.30G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/213 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/782 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

added_tokens.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/817 [00:00<?, ?B/s]

✅ Model loaded and configured!


In [5]:

dataset = load_dataset(DATASET_NAME)
print(dataset)

def convert_sample(sample):
    return {
        "messages": [
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": sample["user_text"]},
                    {"type": "image", "image": sample["image"]}
                ]
            },
            {
                "role": "assistant",
                "content": [
                    {"type": "text", "text": sample["assistant_text"]}
                ]
            }
        ]
    }

train_data = [convert_sample(s) for s in dataset["train"]]
print(f"✅ Converted {len(train_data)} samples")

README.md:   0%|          | 0.00/550 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/46.9M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/414 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['image', 'user_text', 'assistant_text', 'filename', 'part_type', 'view_type', 'quality', 'confidence'],
        num_rows: 414
    })
})
✅ Converted 414 samples


In [6]:
data_collator = UnslothVisionDataCollator(
    model, tokenizer,
    train_on_responses_only=True,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    max_steps=200,
    learning_rate=2e-4,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=10,
    save_steps=50,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="linear",
    warmup_steps=10,
    seed=3407,
    report_to="wandb",
    dataset_text_field="",
    dataset_kwargs={"skip_prepare_dataset": True},
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    data_collator=data_collator,
    train_dataset=train_data,
    args=training_args,
)


Unsloth: Model does not have a default image size - using 512


In [7]:
print("🚀 Starting training...")
trainer.train()
print("✅ Training complete!")

The model is already on multiple devices. Skipping the move to device specified in `args`.


🚀 Starting training...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 414 | Num Epochs = 4 | Total steps = 200
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 39,321,600 of 4,477,137,408 (0.88% trained)
wandb: Currently logged in as: ordinarypeoplecompany (ordinarypeoplecompany-dr-m-g-r-educational-and-research-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: Detected [huggingface_hub.inference, openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Weave is installed but not imported. Add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai/


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
10,2.018900
20,0.885200
30,0.479900
40,0.397300
50,0.371800
60,0.313300
70,0.300700
80,0.285600
90,0.280200
100,0.272000


train/epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇███
train/global_step,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇███
train/grad_norm,█▆▃▂▁▃▁▂▂▂▁▂▂▃▂▂▃▃▂▃
train/learning_rate,███▇▇▆▆▆▅▅▄▄▄▃▃▃▂▂▁▁
train/loss,█▄▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
total_flos,2.1428550839322624e+16
train/epoch,3.85024
train/global_step,200
train/grad_norm,0.42946
train/learning_rate,0.0
train/loss,0.2034


✅ Training complete!


In [8]:
model.save_pretrained(f"{OUTPUT_DIR}/lora")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/lora")
print(f"✅ Saved to: {OUTPUT_DIR}/lora")

✅ Saved to: ./counterfeit_detector/lora


In [9]:
FastVisionModel.for_inference(model)

# Test sample
test_sample = dataset["train"][0]
test_image = test_sample["image"]
instruction = "Analyze this motorcycle part image for quality assessment."

messages = [
    {"role": "user", "content": [
        {"type": "image"},
        {"type": "text", "text": instruction}
    ]}
]

input_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
inputs = tokenizer(test_image, input_text, add_special_tokens=False, return_tensors="pt").to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt=True)

print("📋 Model Output:")
_ = model.generate(**inputs, streamer=text_streamer, max_new_tokens=512,
                   temperature=0.7, min_p=0.1, use_cache=True)


📋 Model Output:
**Quality Assessment Report**

**Helmet Shell:** HIGH_QUALITY
**Visor Quality:** NOT_VISIBLE
**Strap/Harness:** NOT_VISIBLE
**Interior Padding:** NOT_VISIBLE
**Brandings/Stickers:** NOT_VISIBLE
**Visible Defects:** None detected

**Overall Assessment:**
• **Quality Rating:** HIGH
• **Confidence:** 90%

**Reasoning:** The visible exterior of the helmet shell appears smooth and free of manufacturing flaws like flash or molding inconsistencies. No defects are visible in the highlighted area. However, the visor, strap, interior padding, and brandings are not visible in this close-up view, limiting full assessment. The red circle highlights a non-defective area, reinforcing the assessment of the visible shell quality.<|im_end|>


In [10]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()

model.push_to_hub_merged(
    "barryallen16/counterfeit-detector-qwen3vl",
    tokenizer,  
    token = user_secrets.get_secret("WRITE_HF_TOKEN"),
    save_method="lora"
)

config.json: 0.00B [00:00, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

...

Unsloth: Merging weights into 16bit:  50%|█████     | 1/2 [01:29<01:29, 89.24s/it]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [02:39<00:00, 79.92s/it]


Unsloth: Merge process complete. Saved to `/kaggle/working/barryallen16/counterfeit-detector-qwen3vl`
